# Make land cover / geomedian side by side animation

In [1]:
import datacube
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import os

import glob
import imageio
import numpy as np
from PIL import Image

from odc.ui import with_ui_cbk
from datacube.utils.cog import write_cog

import sys
sys.path.insert(1, "../Tools/")
from dea_tools.plotting import rgb, display_map
from dea_tools.landcover import plot_land_cover

from matplotlib.colors import ListedColormap
from matplotlib import colors as mcolours

In [2]:
level4_cmap = {0: (255, 255, 255, 255, "No Data"),
               1: (151, 187, 26, 255, 'Cultivated Terrestrial\n Vegetated:'),
               2: (151, 187, 26, 255, 'Cultivated Terrestrial\n Vegetated: Woody'),
               3: (209, 224, 51, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous'),
               4: (197, 168, 71, 255, 'Cultivated Terrestrial\n Vegetated: Closed\n (> 65 %)'),
               5: (205, 181, 75, 255, 'Cultivated Terrestrial\n Vegetated: Open\n (40 to 65 %)'),
               6: (213, 193, 79, 255, 'Cultivated Terrestrial\n Vegetated: Open\n (15 to 40 %)'),
               7: (228, 210, 108, 255, 'Cultivated Terrestrial\n Vegetated: Sparse\n (4 to 15 %)'),
               8: (242, 227, 138, 255, 'Cultivated Terrestrial\n Vegetated: Scattered\n (1 to 4 %)'),
               # 9: (197, 168, 71, 255, 'Cultivated Terrestrial\n Vegetated: Woody Closed\n (> 65 %)'),
               # 10: (205, 181, 75, 255, 'Cultivated Terrestrial\n Vegetated: Woody Open\n (40 to 65 %)'),
               # 11: (213, 193, 79, 255, 'Cultivated Terrestrial\n Vegetated: Woody Open\n (15 to 40 %)'),
               # 12: (228, 210, 108, 255, 'Cultivated Terrestrial\n Vegetated: Woody Sparse\n (4 to 15 %)'),
               # 13: (242, 227, 138, 255, 'Cultivated Terrestrial\n Vegetated: Woody Scattered\n (1 to 4 %)'),
               14: (228, 224, 52, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Closed\n (> 65 %)'),
               15: (235, 232, 84, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Open\n (40 to 65 %)'),
               16: (242, 240, 127, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Open\n (15 to 40 %)'),
               17: (249, 247, 174, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Sparse\n (4 to 15 %)'),
               18: (255, 254, 222, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Scattered\n (1 to 4 %)'),
               19: (14, 121, 18, 255, 'Natural Terrestrial Vegetated:'),
               20: (26, 177, 87, 255, 'Natural Terrestrial Vegetated: Woody'),
               21: (94, 179, 31, 255, 'Natural Terrestrial Vegetated: Herbaceous'),
               22: (14, 121, 18, 255, 'Natural Terrestrial Vegetated: Closed (> 65 %)'),
               23: (45, 141, 47, 255, 'Natural Terrestrial Vegetated: Open (40 to 65 %)'),
               24: (80, 160, 82, 255, 'Natural Terrestrial Vegetated: Open (15 to 40 %)'),
               25: (117, 180, 118, 255, 'Natural Terrestrial Vegetated: Sparse (4 to 15 %)'),
               26: (154, 199, 156, 255, 'Natural Terrestrial Vegetated: Scattered (1 to 4 %)'),
               27: (14, 121, 18, 255, 'Natural Terrestrial Vegetated: Woody Closed (> 65 %)'),
               28: (45, 141, 47, 255, 'Natural Terrestrial Vegetated: Woody Open (40 to 65 %)'),
               29: (80, 160, 82, 255, 'Natural Terrestrial Vegetated: Woody Open (15 to 40 %)'),
               30: (117, 180, 118, 255, 'Natural Terrestrial Vegetated: Woody Sparse (4 to 15 %)'),
               31: (154, 199, 156, 255, 'Natural Terrestrial Vegetated: Woody Scattered (1 to 4 %)'),
               32: (119, 167, 30, 255, 'Natural Terrestrial Vegetated: Herbaceous Closed (> 65 %)'),
               33: (136, 182, 51, 255, 'Natural Terrestrial Vegetated: Herbaceous Open (40 to 65 %)'),
               34: (153, 196, 80, 255, 'Natural Terrestrial Vegetated: Herbaceous Open (15 to 40 %)'),
               35: (170, 212, 113, 255, 'Natural Terrestrial Vegetated: Herbaceous Sparse (4 to 15 %)'),
               36: (186, 226, 146, 255, 'Natural Terrestrial Vegetated: Herbaceous Scattered (1 to 4 %)'),
               # 37: (86, 236, 231, 255, 'Cultivated Aquatic Vegetated:'),
               # 38: (61, 170, 140, 255, 'Cultivated Aquatic Vegetated: Woody'),
               # 39: (82, 231, 172, 255, 'Cultivated Aquatic Vegetated: Herbaceous'),
               # 40: (43, 210, 203, 255, 'Cultivated Aquatic Vegetated: Closed (> 65 %)'),
               # 41: (73, 222, 216, 255, 'Cultivated Aquatic Vegetated: Open (40 to 65 %)'),
               # 42: (110, 233, 228, 255, 'Cultivated Aquatic Vegetated: Open (15 to 40 %)'),
               # 43: (149, 244, 240, 255, 'Cultivated Aquatic Vegetated: Sparse (4 to 15 %)'),
               # 44: (187, 255, 252, 255, 'Cultivated Aquatic Vegetated: Scattered (1 to 4 %)'),
               # 45: (43, 210, 203, 255, 'Cultivated Aquatic Vegetated: Woody Closed (> 65 %)'),
               # 46: (73, 222, 216, 255, 'Cultivated Aquatic Vegetated: Woody Open (40 to 65 %)'),
               # 47: (110, 233, 228, 255, 'Cultivated Aquatic Vegetated: Woody Open (15 to 40 %)'),
               # 48: (149, 244, 240, 255, 'Cultivated Aquatic Vegetated: Woody Sparse (4 to 15 %)'),
               # 49: (187, 255, 252, 255, 'Cultivated Aquatic Vegetated: Woody Scattered (1 to 4 %)'),
               # 50: (82, 231, 196, 255, 'Cultivated Aquatic Vegetated: Herbaceous Closed (> 65 %)'),
               # 51: (113, 237, 208, 255, 'Cultivated Aquatic Vegetated: Herbaceous Open (40 to 65 %)'),
               # 52: (144, 243, 220, 255, 'Cultivated Aquatic Vegetated: Herbaceous Open (15 to 40 %)'),
               # 53: (175, 249, 232, 255, 'Cultivated Aquatic Vegetated: Herbaceous Sparse (4 to 15 %)'),
               # 54: (207, 255, 244, 255, 'Cultivated Aquatic Vegetated: Herbaceous Scattered (1 to 4 %)'),
               55: (30, 191, 121, 255, 'Natural Aquatic Vegetated:'),
               56: (18, 142, 148, 255, 'Natural Aquatic Vegetated: Woody'),
               57: (112, 234, 134, 255, 'Natural Aquatic Vegetated: Herbaceous'),
               58: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Closed (> 65 %)'),
               59: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Open (40 to 65 %)'),
               60: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Open (15 to 40 %)'),
               61: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Sparse (4 to 15 %)'),
               62: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Scattered (1 to 4 %)'),
               63: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %)'),
               64: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %) Water > 3 months (semi-) permenant'),
               65: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %) Water < 3 months (temporary or seasonal)'),
               66: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %)'),
               67: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %) Water > 3 months (semi-) permenant'),
               68: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %) Water < 3 months (temporary or seasonal)'),
               69: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %)'),
               70: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %) Water > 3 months (semi-) permenant'),
               71: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %) Water < 3 months (temporary or seasonal)'),
               72: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %)'),
               73: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %) Water > 3 months (semi-) permenant'),
               74: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %) Water < 3 months (temporary or seasonal)'),
               75: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %)'),
               76: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %) Water > 3 months (semi-) permenant'),
               77: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %) Water < 3 months (temporary or seasonal)'),
               78: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %)'),
               79: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %) Water > 3 months (semi-) permenant'),
               80: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %) Water < 3 months (temporary or seasonal)'),
               81: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %)'),
               82: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %) Water > 3 months (semi-) permenant'),
               83: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %) Water < 3 months (temporary or seasonal)'),
               84: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %)'),
               85: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %) Water > 3 months (semi-) permenant'),
               86: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %) Water < 3 months (temporary or seasonal)'),
               87: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %)'),
               88: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %) Water > 3 months (semi-) permenant'),
               89: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %) Water < 3 months (temporary or seasonal)'),
               90: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %)'),
               91: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %) Water > 3 months (semi-) permenant'),
               92: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %) Water < 3 months (temporary or seasonal)'),
               93: (218, 92, 105, 255, 'Artificial Surface:'),
               94: (243, 171, 105, 255, 'Natural Surface:'),
               95: (255, 230, 140, 255, 'Natural Surface: Sparsely vegetated'),
               96: (250, 210, 110, 255, 'Natural Surface: Very sparsely vegetated'),
               97: (243, 171, 105, 255, 'Natural Surface: Bare areas, unvegetated'),
               98: (77, 159, 220, 255, 'Water:'),
               99: (77, 159, 220, 255, 'Water: (Water)'),
               100: (187, 220, 233, 255, 'Water: (Water) Tidal area'),
               101: (27, 85, 186, 255, 'Water: (Water) Perennial (> 9 months)'),
               102: (52, 121, 201, 255, 'Water: (Water) Non-perennial (7 to 9 months)'),
               103: (79, 157, 217, 255, 'Water: (Water) Non-perennial (4 to 6 months)'),
               104: (133, 202, 253, 255, 'Water: (Water) Non-perennial (1 to 3 months)'),
               # 105: (250, 250, 250, 255, 'Water: (Snow)')
               }

In [3]:
def load_landcover_data(measurements, lon_range, lat_range, year):
    """
    Load geomedian dataset for a given year.

    Parameters:
    - measurements (list): list of measurement names.
    - lon_range (tuple): longitude range.
    - lat_range (tuple): latitude range.
    - year (int): year for which data is to be loaded.

    Returns:
    - geomedian (xarray.Dataset): Loaded geomedian dataset.
    """
    dc = datacube.Datacube(app="")
    landcover = dc.load(
        product="ga_ls_landcover_class_cyear_2",
        measurements=measurements,
        x=lon_range,
        y=lat_range,
        time=(str(year)),
    )
    return landcover

In [4]:
def get_product_for_year(year):
    """
    Get the product name based on the given year.

    Parameters:
    - year (int): year for which product is needed.

    Returns:
    - product (str): product name.
    """
    if 1988 <= year <= 1999:
        return "ga_ls5t_gm_cyear_3"
    elif 2000 <= year <= 2003:
        return "ga_ls7e_gm_cyear_3"
    elif 2004 <= year <= 2007:
        return "ga_ls5t_gm_cyear_3"
    elif year == 2008:
        return "ga_ls7e_gm_cyear_3"
    elif 2009 <= year <= 2011:
        return "ga_ls5t_gm_cyear_3"
    elif year == 2012:
        return "ga_ls7e_gm_cyear_3"
    elif year >= 2013:
        return "ga_ls8cls9c_gm_cyear_3"
    else:
        return None
    
def load_geomedian_data(product, measurements, lon_range, lat_range, year):
    """
    Load geomedian dataset for a given year.

    Parameters:
    - product (str): product name.
    - measurements (list): list of measurement names.
    - lon_range (tuple): longitude range.
    - lat_range (tuple): latitude range.
    - year (int): year for which data is to be loaded.

    Returns:
    - geomedian (xarray.Dataset): Loaded geomedian dataset.
    """
    dc = datacube.Datacube(app="")
    geomedian = dc.load(
        product=product,
        measurements=measurements,
        x=lon_range,
        y=lat_range,
        time=(str(year), str(year)),
    )
    return geomedian

In [5]:
def plot_layer(colours, data, ax=None):
    colour_arr = []

    for key, value in colours.items():
        colour_arr.append(np.array(value[:-2]) / 255)


    cmap = mcolours.ListedColormap(colour_arr)
    bounds = list(colours)
    bounds.append(255)
    norm = mcolours.BoundaryNorm(np.array(bounds) - 0.1, cmap.N)

    # Plot the provided layer
    im = data.isel(time=0).plot(cmap=cmap, norm=norm, add_colorbar=False)

    return im

In [6]:
def make_gif(frame_folder, start_year, end_year, location):
    """
    Create a gif from a folder of images (png). 
    Parameters:
    - frame folder (str): path to the folder containing the images. 
    - start_year (int): starting year for the GIF title.
    - end_year (int): ending year for the GIF title.
    - location (str): location for the GIF title.
    Returns:
    - the gif will be saved in the current folder. 
    """
    frames = [Image.open(image) for image in sorted(glob.glob(f"{frame_folder}/*.png"))]
    frame_one = frames[0]
    frame_one.save(f"Geomedian and Land Cover {start_year}-{end_year} {location}.gif", format="GIF", append_images=frames,
               save_all=True, duration=500, loop=0, optimize=True)

In [7]:
def delete_generated_images(folder):
    """
    Delete all the images from the 'images' directory.

    Parameters:
    - folder (str): name of folder/folder path

    Returns:
    - None
    """
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")

In [8]:
# location = "Bauxite Mine, WA"
# lat = -32.5549
# lon = 116.1667
# lat_buffer = 0.2
# lon_buffer = 0.3

# location = "North Brisbane, Qld"
# lat = -27.2731
# lon = 152.9970
# lat_buffer = 0.1
# lon_buffer = 0.15

location = "Canberra, ACT"
lat = -35.3062
lon =  149.1216
lat_buffer = 0.1
lon_buffer = 0.1

lat_range = (lat - lat_buffer, lat + lat_buffer)
lon_range = (lon - lon_buffer, lon + lon_buffer)

In [9]:
display_map(x=lon_range, y=lat_range)

In [12]:
def make_plots(start_year, end_year):
    for year in range(start_year, end_year + 1):
        # load land cover 
        lc_bands = ["level4"]
        landcover = load_landcover_data(lc_bands, lon_range, lat_range, year)
        # load geomedian
        gm_bands = ["nbart_red", "nbart_green", "nbart_blue"]
        product = get_product_for_year(year)
        geomedian = load_geomedian_data(product, gm_bands, lon_range, lat_range, year)
        f, axarr = plt.subplots(1, 2, figsize=(12, 6), squeeze=False)
        # plot geomedian
        rgb(
            geomedian,
            bands=["nbart_red", "nbart_green", "nbart_blue"],
            ax=axarr[0, 0],
            robust=True,
        )
        axarr[0, 0].set_title("DEA Geomedian")  
        axarr[0, 0].text(0.97, 0.97, year, transform=axarr[0, 0].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))
        # plot land cover
        plot_layer(level4_cmap, landcover.level4, ax=axarr[0, 1]) 
        axarr[0, 1].set_title("DEA Land Cover")
        axarr[0, 1].text(0.97, 0.97, year, transform=axarr[0, 1].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))
        # remove uggo text and stuff 
        axarr[0, 0].set_xlabel('')
        axarr[0, 0].set_ylabel('')
        axarr[0, 0].set_xticks([])
        axarr[0, 0].set_yticks([])
        axarr[0, 1].set_xlabel('')
        axarr[0, 1].set_ylabel('')
        axarr[0, 1].set_xticks([])
        axarr[0, 1].set_yticks([])

        plt.tight_layout()  

        plt.savefig(f'images/{year}_{location}.png', dpi=150)

        # plt.show()
        plt.close()


In [ ]:
make_plots(1988, 2020)

In [117]:
make_gif("images", 1988, 2020, location)

In [118]:
delete_generated_images("images")

In [78]:
lc_bands = ["level4"]
landcover = load_landcover_data(lc_bands, lon_range, lat_range, 2015)
# lc

In [79]:
gm_bands = ["nbart_red", "nbart_green", "nbart_blue"]
product = get_product_for_year(year)
geomedian = load_geomedian_data(product, gm_bands, lon_range, lat_range, year)
# geomedian

In [ ]:
f, axarr = plt.subplots(1, 2, figsize=(12, 6), squeeze=False)

rgb(
    geomedian,
    bands=["nbart_red", "nbart_green", "nbart_blue"],
    ax=axarr[0, 0],
    robust=True,
)
# axarr[0, 0].set_title("DEA Geomedian")  
# axarr[0, 0].set_title("")  
axarr[0, 0].text(0.97, 0.97, year, transform=axarr[0, 0].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))

plot_layer(level4_cmap, landcover.level4, ax=axarr[0, 1]) 
# axarr[0, 1].set_title("DEA Land Cover")
axarr[0, 1].set_title("")
axarr[0, 1].text(0.97, 0.97, year, transform=axarr[0, 1].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))


axarr[0, 0].set_xlabel('')
axarr[0, 0].set_ylabel('')
axarr[0, 0].set_xticks([])
axarr[0, 0].set_yticks([])
axarr[0, 1].set_xlabel('')
axarr[0, 1].set_ylabel('')
axarr[0, 1].set_xticks([])
axarr[0, 1].set_yticks([])

plt.tight_layout()  

plt.savefig(f'images/{year}_{location}.png', dpi=150)

plt.show()